# 去中心化联邦学习（DFL）结果分析

用于浏览全部 DFL 结果、展示单次运行、比较多个算法，以及比较单算法的多组参数。notebook 直接使用 `core.result` 查询数据库并读取逐轮指标。

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (current, *current.parents) if (path / "pyproject.toml").is_file()
)
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from core.result import ResultsReader, RunQuery
from utils.plots import (
    algorithm_names, configure_plots, plot_algorithm_comparison,
    plot_algorithm_runs, plot_parameter_comparison, plot_run, runs_table,
)

configure_plots()
DFL_ALGORITHMS = algorithm_names(PROJECT_ROOT / "algo" / "dfl")
READER = ResultsReader()
RUN_IDS = None
RUN_FILTERS = {}
RUN_QUERY = RunQuery(
    run_ids=(RUN_IDS,) if isinstance(RUN_IDS, str) else tuple(RUN_IDS) if RUN_IDS is not None else None,
    algorithms=tuple(DFL_ALGORITHMS),
    statuses=("completed",),
    args=RUN_FILTERS,
    apply_defaults=False,
)
DFL_RUNS = READER.query_runs(RUN_QUERY)
if RUN_IDS is None:
    latest_by_configuration = {}
    for run in reversed(DFL_RUNS):
        configuration = {
            name: value for name, value in run.args.items()
            if name not in {"algo", "data_root", "devices", "service_device", "log_level", "result_root", "times", "trials"}
        }
        key = (run.algorithm, json.dumps(configuration, sort_keys=True, ensure_ascii=False))
        latest_by_configuration[key] = run
    DFL_RUNS = sorted(
        latest_by_configuration.values(),
        key=lambda run: (run.started_at, run.run_id),
        reverse=True,
    )
DFL_METRICS = READER.read_metrics_many(run.run_id for run in DFL_RUNS)
print(f"DFL 运行 {len(DFL_RUNS)} 次。")
print(dict(sorted(Counter(run.algorithm for run in DFL_RUNS).items())))

## 1. 结果概览与筛选

可以按运行属性、实验参数或拓扑参数筛选。

In [ ]:
display(pd.DataFrame(runs_table(DFL_RUNS, DFL_METRICS, limit=30)))

## 2. 单次运行或单个算法展示

In [ ]:
# 默认列表按时间倒序；第一个是最新匹配记录。
RUN = DFL_RUNS[0]
RUN_METRICS = {RUN.run_id: DFL_METRICS[RUN.run_id]}
display(pd.Series({
    "run_id": RUN.run_id,
    "algorithm": RUN.algorithm,
    "status": RUN.status,
    "run_path": str(RUN.run_path),
    "args": RUN.args,
}))
plot_run(RUN, RUN_METRICS)

# 直接用新查询接口按具体参数筛选，然后读取对应指标：
# selected = READER.query_runs(RunQuery(
#     algorithms=("dfedavgm",),
#     statuses=("completed",), args={"dataset": "cifar10"}, apply_defaults=False,
# ))
# selected_metrics = READER.read_metrics_many(run.run_id for run in selected)
# plot_algorithm_runs(selected, selected_metrics, "dfedavgm")

## 3. 多算法对比

请特别固定 `adj_type`、`edge_p`、`k_small_world`、`m_scale_free` 等拓扑控制变量。

In [ ]:
COMPARE_ALGORITHMS = ('dfedavgm', 'dfedset', 'dispfl')
COMPARE_RUNS = READER.query_runs(RunQuery(
    algorithms=COMPARE_ALGORITHMS,
    statuses=("completed",),
    args={"dataset": "cifar10", "partition": "dirichlet", "dir": 0.1},
    apply_defaults=False,
))
latest_by_configuration = {}
for run in reversed(COMPARE_RUNS):
    configuration = {
        name: value for name, value in run.args.items()
        if name not in {"algo", "data_root", "devices", "service_device", "log_level", "result_root", "times", "trials"}
    }
    key = (run.algorithm, json.dumps(configuration, sort_keys=True, ensure_ascii=False))
    latest_by_configuration[key] = run
COMPARE_RUNS = list(latest_by_configuration.values())
COMPARE_METRICS = READER.read_metrics_many(run.run_id for run in COMPARE_RUNS)
fig, axis, compared = plot_algorithm_comparison(
    COMPARE_RUNS, COMPARE_METRICS, COMPARE_ALGORITHMS, metric="accuracy"
)

## 4. 单算法参数对比

可同时比较算法参数与拓扑参数的组合。

In [ ]:
PARAM_ALGORITHM = "dfedavgm"
PARAMETERS = ['adj_type', 'edge_p']
PARAM_RUNS = READER.query_runs(RunQuery(
    algorithms=(PARAM_ALGORITHM,),
    statuses=("completed",),
    args={"dataset": "cifar10"},
    apply_defaults=False,
))
latest_by_configuration = {}
for run in reversed(PARAM_RUNS):
    configuration = {
        name: value for name, value in run.args.items()
        if name not in {"algo", "data_root", "devices", "service_device", "log_level", "result_root", "times", "trials"}
    }
    key = (run.algorithm, json.dumps(configuration, sort_keys=True, ensure_ascii=False))
    latest_by_configuration[key] = run
PARAM_RUNS = list(latest_by_configuration.values())
PARAM_METRICS = READER.read_metrics_many(run.run_id for run in PARAM_RUNS)
fig, axis, groups = plot_parameter_comparison(
    PARAM_RUNS, PARAM_METRICS, PARAM_ALGORITHM, PARAMETERS, metric="accuracy"
)

## 5. 日志与 checkpoint

In [ ]:
RUN = DFL_RUNS[0]
log_path = RUN.run_path / "run.log"
print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-20:]))
checkpoint_path = RUN.run_path / "checkpoints" / f"round_{RUN.latest_round}.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)